In [1]:
from pathlib import Path
from pypdf import PdfReader

PIN_PDF_PATH = Path("../data/raw/raj_pin.pdf")

reader = PdfReader(PIN_PDF_PATH)

print("Total PDF pages:", len(reader.pages))

first_page_text = reader.pages[0].extract_text()

print("\nFirst page text:")
print(first_page_text[:5000])

Total PDF pages: 323

First page text:
Office Name Pincode Delivery/
Non Delivery
Office 
Type
Circle Region Division
Ajmer H.O 305001 Delivery HO Rajasthan Circle Ajmer Region Ajmer Division
Alwar Gate Ajmer S.O 305001 Non-Delivery PO Rajasthan Circle Ajmer Region Ajmer Division
Ashok Marg Ajmer S.O 305001 Non-Delivery PO Rajasthan Circle Ajmer Region Ajmer Division
Bhajan Ganj Ajmer S.O 305001 Non-Delivery PO Rajasthan Circle Ajmer Region Ajmer Division
Dargha Shareef Ajmer S.O 305001 Non-Delivery PO Rajasthan Circle Ajmer Region Ajmer Division
Dhan Mandi Ajmer S.O 305001 Non-Delivery PO Rajasthan Circle Ajmer Region Ajmer Division
Ganj Ajmer S.O 305001 Non-Delivery PO Rajasthan Circle Ajmer Region Ajmer Division
Gujar Dharti Ajmer S.O 305001 Non-Delivery PO Rajasthan Circle Ajmer Region Ajmer Division
Jln Hospital Ajmer S.O 305001 Non-Delivery PO Rajasthan Circle Ajmer Region Ajmer Division
Jones Ganj Ajmer S.O 305001 Non-Delivery PO Rajasthan Circle Ajmer Region Ajmer Division
Kais

In [2]:
import re
import pandas as pd
from pathlib import Path
from pypdf import PdfReader

# --------------------------------------------------
# PATH
# --------------------------------------------------

PIN_PDF_PATH = Path("../data/raw/raj_pin.pdf")
OUTPUT_PATH = Path("../data/processed/rajasthan_pincode_directory.csv")

# --------------------------------------------------
# READ ALL PDF PAGES
# --------------------------------------------------

reader = PdfReader(PIN_PDF_PATH)

all_text = []

for page in reader.pages:
    text = page.extract_text()
    if text:
        all_text.append(text)

full_text = "\n".join(all_text)

print("Total pages:", len(reader.pages))
print("Extracted characters:", len(full_text))

Total pages: 323
Extracted characters: 870599


In [3]:
# --------------------------------------------------
# EXTRACT PIN CODE RECORDS
# --------------------------------------------------

pattern = re.compile(
    r"(.+?)\s+"
    r"(\d{6})\s+"
    r"(Delivery|Non-Delivery)\s+"
    r"([A-Za-z]+)\s+"
    r"Rajasthan\s+[Cc]ircle\s+"
    r"(.+?)\s+"
    r"(.+?)\s+"
    r"Division"
)

matches = pattern.findall(full_text)

print("Records extracted:", len(matches))

records = []

for match in matches:
    office_name = match[0].strip()
    pincode = match[1].strip()
    delivery_status = match[2].strip()
    office_type = match[3].strip()
    region = match[4].strip()
    division = match[5].strip()

    records.append({
        "office_name": office_name,
        "pincode": pincode,
        "delivery_status": delivery_status,
        "office_type": office_type,
        "region": region,
        "division": division
    })

pin_df = pd.DataFrame(records)

print(pin_df.shape)
print(pin_df.head(20))

Records extracted: 10311
(10311, 6)
                  office_name pincode delivery_status office_type region  \
0                   Ajmer H.O  305001        Delivery          HO  Ajmer   
1        Alwar Gate Ajmer S.O  305001    Non-Delivery          PO  Ajmer   
2        Ashok Marg Ajmer S.O  305001    Non-Delivery          PO  Ajmer   
3       Bhajan Ganj Ajmer S.O  305001    Non-Delivery          PO  Ajmer   
4    Dargha Shareef Ajmer S.O  305001    Non-Delivery          PO  Ajmer   
5        Dhan Mandi Ajmer S.O  305001    Non-Delivery          PO  Ajmer   
6              Ganj Ajmer S.O  305001    Non-Delivery          PO  Ajmer   
7      Gujar Dharti Ajmer S.O  305001    Non-Delivery          PO  Ajmer   
8      Jln Hospital Ajmer S.O  305001    Non-Delivery          PO  Ajmer   
9        Jones Ganj Ajmer S.O  305001    Non-Delivery          PO  Ajmer   
10      Kaiser Ganj Ajmer S.O  305001    Non-Delivery          PO  Ajmer   
11       Kutchery S.O (Ajmer)  305001    Non-Deliver

In [4]:
# --------------------------------------------------
# CLEAN DATA
# --------------------------------------------------

pin_df = pin_df.drop_duplicates()

pin_df["pincode"] = (
    pin_df["pincode"]
    .astype(str)
    .str.strip()
)

pin_df["office_name"] = (
    pin_df["office_name"]
    .astype(str)
    .str.strip()
)

print("Final shape:", pin_df.shape)
print("Unique PIN codes:", pin_df["pincode"].nunique())
print("Missing values:")
print(pin_df.isnull().sum())

Final shape: (10311, 6)
Unique PIN codes: 1001
Missing values:
office_name        0
pincode            0
delivery_status    0
office_type        0
region             0
division           0
dtype: int64


In [5]:
# --------------------------------------------------
# SAVE PIN DATABASE
# --------------------------------------------------

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

pin_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "PIN database saved successfully!"
)

print(
    "Path:",
    OUTPUT_PATH
)

print(
    "Shape:",
    pin_df.shape
)

PIN database saved successfully!
Path: ..\data\processed\rajasthan_pincode_directory.csv
Shape: (10311, 6)


In [6]:
pin_df[pin_df["pincode"] == "341501"]

,office_name,pincode,delivery_status,office_type,region,division
8172,Badu S.O (Nagaur),341501,Delivery,PO,Jodhpur,Region Nagaur
8173,Bhadawa B.O,341501,Delivery,BO,Jodhpur,Region Nagaur
8174,Gular B.O,341501,Delivery,BO,Jodhpur,Region Nagaur
8175,Harnawa B.O,341501,Delivery,BO,Jodhpur,Region Nagaur
8176,Janjila B.O,341501,Delivery,BO,Jodhpur,Region Nagaur
8177,Lalana Khurd B.O,341501,Delivery,BO,Jodhpur,Region Nagaur
8178,Nimbri B.O,341501,Delivery,BO,Jodhpur,Region Nagaur
8179,Sirsu B.O,341501,Delivery,BO,Jodhpur,Region Nagaur


In [7]:
# Check PIN database
print("Total records:", len(pin_df))
print("Unique PIN codes:", pin_df["pincode"].nunique())

print("\n341501 records:")
print(
    pin_df[
        pin_df["pincode"] == "341501"
    ].to_string(index=False)
)

Total records: 10311
Unique PIN codes: 1001

341501 records:
      office_name pincode delivery_status office_type  region      division
Badu S.O (Nagaur)  341501        Delivery          PO Jodhpur Region Nagaur
      Bhadawa B.O  341501        Delivery          BO Jodhpur Region Nagaur
        Gular B.O  341501        Delivery          BO Jodhpur Region Nagaur
      Harnawa B.O  341501        Delivery          BO Jodhpur Region Nagaur
      Janjila B.O  341501        Delivery          BO Jodhpur Region Nagaur
 Lalana Khurd B.O  341501        Delivery          BO Jodhpur Region Nagaur
       Nimbri B.O  341501        Delivery          BO Jodhpur Region Nagaur
        Sirsu B.O  341501        Delivery          BO Jodhpur Region Nagaur
